

## ¿Cómo funciona DVC con Almacenamiento Local?

En lugar de subir datos a la nube, DVC utilizará un directorio dentro de tu máquina, disco duro externo o carpeta compartida de red (NAS/NFS).

```text
  [ Tu Código OOP / Scripts / .dvc ] ----> Se guardan en -------> Git (GitHub / GitLab)
  [ Datasets / Imágenes / Modelos ] -----> Se guardan en -------> DVC Remote Local (/mnt/dvc_storage)
```

--- 

## Estructura del Proyecto (Cookiecutter Data Science)

```text
.
├── .git/                    <-- Repositorio de Git
├── .gitignore               <-- Editado por DVC para ignorar datos pesados
├── .dvc/                    <-- Configuración interna de DVC
│   └── config               <-- Define la ruta del almacenamiento local
├── params.yaml              <-- Hiperparámetros (rastreado por Git)
├── dvc.yaml                <-- Pipeline de ML (rastreado por Git)
├── dvc.lock                <-- Estado exacto del pipeline (rastreado por Git)
├── data/
│   ├── processed/           <-- Datos limpios procesados
│   └── raw/                 <-- Datasets pesados en bruto (CSVs, imágenes)
│       ├── transactions.csv.dvc   <-- Puntero pequeño (rastreado por Git)
│       └── sample_images.dvc      <-- Puntero pequeño (rastreado por Git)
├── metrics/                 <-- Métricas en JSON y gráficos (rastreado por Git)
├── notebooks/
│   └── tutorial_dvc_local.ipynb <-- Este cuaderno interactivo
└── src/
    ├── data/
    │   └── make_dataset.py  <-- Clase `DataProcessor`
    └── models/
        └── train_model.py   <-- Clase `ModelTrainer`
```

## Paso 1: Instalación e Inicialización de Git + DVC

In [ ]:
# Instalación de DVC base (no requiere plugins de nube) y librerías de ML
!pip install dvc pandas scikit-learn pyyaml pillow

In [1]:
# Ajuste del directorio de trabajo a la raíz del proyecto si estamos en notebooks/
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    %cd ..
print(f"Directorio de trabajo actual: {os.getcwd()}")

Directorio de trabajo actual: e:\Python\Practicas_ML


In [2]:
# 1. Inicializar Git y DVC
!git init
!dvc init

# 2. Guardar la configuración inicial en Git
!git add .dvc/ .gitignore
!git commit -m "build: initialize DVC and Git environment"

Reinitialized existing Git repository in E:/Python/Practicas_ML/.git/


ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.
fatal: pathspec '.gitignore' did not match any files


On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	1 Seleccion de variables.ipynb
	AdvancedDVC.ipynb
	AdvancedMLFlow.ipynb
	BasicMLFlow.ipynb
	aml_project/
	archivos/
	local_dvc.ipynb

nothing added to commit but untracked files present (use "git add" to track)


## Paso 2: Configuración del Remote Local en DVC

Puedes usar cualquier ruta fuera del proyecto como remote (un directorio local, disco externo o ruta compartida de red).

In [3]:
# Definir una ruta local fuera del repositorio (ejemplo: ../dvc_storage_remote)
LOCAL_REMOTE_PATH = os.path.abspath("../dvc_storage_remote")
os.makedirs(LOCAL_REMOTE_PATH, exist_ok=True)

# Registrar el remote local en DVC
!dvc remote add -d local_remote {LOCAL_REMOTE_PATH}
!dvc remote list

Setting 'local_remote' as a default remote.
gdrive_remote   gdrive://1kEyy-a0Ij3I0BKFjkn3aTOBu3Qvc_ntn
local_remote    e:\Python\dvc_storage_remote    (default)


In [4]:
# Registrar la configuración de DVC en Git
!git add .dvc/config
!git commit -m "config: set local directory as default DVC remote"

[master 98b937b] config: set local directory as default DVC remote
 1 file changed, 3 insertions(+), 1 deletion(-)


## Paso 3: Versionado de Datos (CSVs e Imágenes) con DVC y Git

In [5]:
import numpy as np
import pandas as pd
from PIL import Image

os.makedirs("data/raw/sample_images", exist_ok=True)

# Generar CSV sintético
df_raw = pd.DataFrame({
    'feature_1': np.random.normal(10, 2, 200),
    'feature_2': np.random.uniform(0, 100, 200),
    'is_fraud': np.random.choice([0, 1], size=200, p=[0.85, 0.15])
})
df_raw.to_csv("data/raw/transactions.csv", index=False)

# Generar Imágenes sintéticas
for i in range(5):
    img_array = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
    Image.fromarray(img_array).save(f"data/raw/sample_images/img_{i+1}.png")

print("✅ Datos pesados generados en data/raw/")

✅ Datos pesados generados en data/raw/


In [6]:
# 1. Rastrear archivos pesados con DVC
!dvc add data/raw/transactions.csv
!dvc add data/raw/sample_images

# 2. Rastrear punteros pequeños (.dvc) y .gitignore con Git
!git add data/raw/transactions.csv.dvc data/raw/sample_images.dvc data/raw/.gitignore
!git commit -m "feat(data): track raw CSV and images directory pointers with DVC"

# 3. Copiar datos al almacenamiento remoto local
!dvc push


To track the changes with git, run:

	git add 'data\raw\transactions.csv.dvc'

To enable auto staging, run:

	dvc config core.autostage true


⠋ Checking graph




To track the changes with git, run:

	git add 'data\raw\sample_images.dvc'

To enable auto staging, run:

	dvc config core.autostage true


⠋ Checking graph



[master d1932b9] feat(data): track raw CSV and images directory pointers with DVC
 2 files changed, 3 insertions(+), 3 deletions(-)
8 files pushed


## Paso 4: Módulos Orientados a Objetos (OOP)

In [7]:
%%writefile params.yaml
prepare:
  split_ratio: 0.25
  random_state: 42

train:
  n_estimators: 150
  max_depth: 6
  target_col: "is_fraud"

Overwriting params.yaml


In [9]:
# Clase de Procesamiento de Datos (src/data/make_dataset.py)
import os

os.makedirs("src/data", exist_ok=True)
with open("src/data/make_dataset.py", "w", encoding="utf-8") as f:
    f.write('''import pandas as pd
import os
import sys

class DataProcessor:
    """Clase responsable de la ingesta y preprocesamiento de datos."""
    def __init__(self, input_path: str, output_path: str):
        self.input_path = input_path
        self.output_path = output_path

    def load_data(self) -> pd.DataFrame:
        if not os.path.exists(self.input_path):
            raise FileNotFoundError(f"El archivo {self.input_path} no existe.")
        return pd.read_csv(self.input_path)

    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        df_clean = df.dropna().copy()
        return df_clean

    def save_data(self, df: pd.DataFrame) -> None:
        os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
        df.to_csv(self.output_path, index=False)
        print(f"✅ [DataProcessor] Datos limpios guardados en: {self.output_path}")

    def run(self) -> None:
        df = self.load_data()
        df_clean = self.clean_data(df)
        self.save_data(df_clean)

if __name__ == '__main__':
    input_file = sys.argv[1] if len(sys.argv) > 1 else 'data/raw/transactions.csv'
    output_file = sys.argv[2] if len(sys.argv) > 2 else 'data/processed/clean_transactions.csv'
    
    processor = DataProcessor(input_file, output_file)
    processor.run()
''')

In [10]:
# Clase de Entrenamiento y Evaluación (src/models/train_model.py)
import os

os.makedirs("src/models", exist_ok=True)
with open("src/models/train_model.py", "w", encoding="utf-8") as f:
    f.write('''import pandas as pd
import yaml
import json
import os
import sys
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

class ModelTrainer:
    """Clase responsable del entrenamiento y evaluación del modelo."""
    def __init__(self, params_path: str, data_path: str, metrics_dir: str):
        self.params = self._load_params(params_path)
        self.data_path = data_path
        self.metrics_dir = metrics_dir
        self.model = None

    def _load_params(self, params_path: str) -> dict:
        with open(params_path, 'r') as f:
            return yaml.safe_load(f)

    def prepare_data(self):
        df = pd.read_csv(self.data_path)
        target_col = self.params['train']['target_col']
        X = df.drop(columns=[target_col])
        y = df[target_col]
        
        return train_test_split(
            X, y, 
            test_size=self.params['prepare']['split_ratio'], 
            random_state=self.params['prepare']['random_state']
        )

    def train(self, X_train, y_train) -> None:
        self.model = RandomForestClassifier(
            n_estimators=self.params['train']['n_estimators'],
            max_depth=self.params['train']['max_depth'],
            random_state=self.params['prepare']['random_state']
        )
        self.model.fit(X_train, y_train)
        print("✅ [ModelTrainer] Modelo entrenado exitosamente.")

    def evaluate(self, X_test, y_test) -> None:
        preds = self.model.predict(X_test)
        metrics = {
            'accuracy': float(accuracy_score(y_test, preds)),
            'precision': float(precision_score(y_test, preds, zero_division=0)),
            'recall': float(recall_score(y_test, preds, zero_division=0)),
            'f1_score': float(f1_score(y_test, preds, zero_division=0))
        }
        
        os.makedirs(self.metrics_dir, exist_ok=True)
        with open(os.path.join(self.metrics_dir, 'eval.json'), 'w') as f:
            json.dump(metrics, f, indent=4)
            
        plots_df = pd.DataFrame({'actual': y_test, 'predicted': preds})
        plots_df.to_csv(os.path.join(self.metrics_dir, 'plots.csv'), index=False)
        print(f"✅ [ModelTrainer] Métricas guardadas en: {self.metrics_dir}/")

    def run(self) -> None:
        X_train, X_test, y_train, y_test = self.prepare_data()
        self.train(X_train, y_train)
        self.evaluate(X_test, y_test)

if __name__ == '__main__':
    trainer = ModelTrainer('params.yaml', 'data/processed/clean_transactions.csv', 'metrics')
    trainer.run()
''')

## Paso 5: Declaración del Pipeline en `dvc.yaml`

In [11]:
%%writefile dvc.yaml
stages:
  preprocess:
    cmd: python src/data/make_dataset.py data/raw/transactions.csv data/processed/clean_transactions.csv
    deps:
      - src/data/make_dataset.py
      - data/raw/transactions.csv
    outs:
      - data/processed/clean_transactions.csv

  train:
    cmd: python src/models/train_model.py
    deps:
      - src/models/train_model.py
      - data/processed/clean_transactions.csv
    params:
      - prepare.split_ratio
      - prepare.random_state
      - train.n_estimators
      - train.max_depth
      - train.target_col
    metrics:
      - metrics/eval.json:
          cache: false
    plots:
      - metrics/plots.csv:
          template: confusion
          x: predicted
          y: actual
          cache: false

Overwriting dvc.yaml


## Paso 6: Ejecución del Pipeline, Sincronización y Experimentos

In [12]:
# Reproducir pipeline completo
!dvc repro

'data\raw\transactions.csv.dvc' didn't change, skipping
Running stage 'preprocess':
> python src/data/make_dataset.py data/raw/transactions.csv data/processed/clean_transactions.csv
✅ [DataProcessor] Datos limpios guardados en: data/processed/clean_transactions.csv
Updating lock file 'dvc.lock'

Running stage 'train':
> python src/models/train_model.py
✅ [ModelTrainer] Modelo entrenado exitosamente.
✅ [ModelTrainer] Métricas guardadas en: metrics/
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


In [13]:
# Guardar en Git los scripts OOP, pipeline, candados y métricas
!git add dvc.yaml dvc.lock params.yaml src/ metrics/ data/processed/.gitignore
!git commit -m "feat(pipeline): OOP pipeline execution with local DVC storage"

# Copiar salidas pesadas al remoto local
!dvc push

[master 6461898] feat(pipeline): OOP pipeline execution with local DVC storage
 5 files changed, 56 insertions(+), 171 deletions(-)
1 file pushed


### Probar un Nuevo Experimento (Variación de Hiperparámetros)

In [14]:
%%writefile params.yaml
prepare:
  split_ratio: 0.30
  random_state: 123

train:
  n_estimators: 300
  max_depth: 10
  target_col: "is_fraud"

Overwriting params.yaml


In [15]:
# Re-ejecutar solo las etapas afectadas
!dvc repro

'data\raw\transactions.csv.dvc' didn't change, skipping
Stage 'preprocess' didn't change, skipping
Running stage 'train':
> python src/models/train_model.py
✅ [ModelTrainer] Modelo entrenado exitosamente.
✅ [ModelTrainer] Métricas guardadas en: metrics/
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


In [16]:
# Comparación de hiperparámetros y métricas con Git
!dvc params diff
!dvc metrics diff HEAD

Path         Param                 HEAD    workspace
params.yaml  prepare.random_state  42      123
params.yaml  prepare.split_ratio   0.25    0.3
params.yaml  train.max_depth       6       10
params.yaml  train.n_estimators    150     300
Path               Metric     HEAD    workspace    Change
metrics\eval.json  accuracy   0.8     0.76667      -0.03333
metrics\eval.json  f1_score   0.0     0.125        0.125
metrics\eval.json  precision  0.0     0.5          0.5
metrics\eval.json  recall     0.0     0.07143      0.07143


In [17]:
# Registrar experimento en Git y DVC local
!git add dvc.lock params.yaml metrics/
!git commit -m "experiment: increase estimators to 300 with max_depth 10"
!dvc push

[master ef1309f] experiment: increase estimators to 300 with max_depth 10
 4 files changed, 30 insertions(+), 20 deletions(-)
Everything is up to date.


## Paso 7: Flujo Colaborativo y Navegación entre Versiones

```bash
# Clonar código desde Git
git clone <URL_REPOSITORIO>
cd mi_proyecto

# Descargar datos desde el almacenamiento local / servidor compartido
dvc pull

# Cambiar de experimento/rama y sincronizar datos
git checkout HEAD~1
dvc checkout
```